In [53]:
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd
from datasets import load_dataset

In [5]:
dataset_path = Path(
    "~/Datasets/Amazon_Reviews/18/meta_Clothing_Shoes_and_Jewelry.json"
).expanduser()
assert dataset_path.exists(), f"Dataset not found at {dataset_path}"

## Utils

In [11]:
class AmazonReviews:
    def __init__(self, dataset_path: Path):
        self.dataset_path = dataset_path

        with open(self.dataset_path, "r") as f:
            self.num_rows = len(f.readlines())

    def __len__(self):
        return self.num_rows

    def load_jsonl_to_df(self, start: int, end: int) -> pd.DataFrame:
        with open(self.dataset_path, "r") as f:
            lines = f.readlines()[start:end]
            lines = [json.loads(line) for line in lines]
            return pd.DataFrame(lines)

In [12]:
# How to load None values as empy not NaN
# df = pd.read_json(dataset_path, lines=True, orient="records", dtype={"related": str})

## Dataset Cleanup

In [106]:
def process_description(description: List[str]) -> str | None:
    if isinstance(description, list):
        cleaned_description = [text for text in description if text != ""]
        return " ".join(cleaned_description)
    elif isinstance(description, str):
        return description
    elif isinstance(description, float) or description is None:
        return None
    else:
        raise ValueError(f"Unexpected type {type(description)}")


def process_category(category: List[str]) -> str | None:
    if isinstance(category, list):
        # if len(category) > 10:
        #     raise ValueError(f"Category too long {category}")
        return category
    elif isinstance(category, float) or category is None:
        return None
    else:
        raise ValueError(f"Unexpected type {type(category)}")


def process_title_brand(title: str) -> str | None:
    if isinstance(title, str):
        return title
    elif isinstance(title, float) or title is None:
        return None
    else:
        raise ValueError(f"Unexpected type {type(title)}")

In [18]:
chunk_size = 100_000

In [59]:
dataset = AmazonReviews(dataset_path)

In [60]:
df = dataset.load_jsonl_to_df(0, 0 + chunk_size)

In [107]:
df["title"] = df["title"].apply(process_title_brand)
df["description"] = df["description"].apply(process_description)
df["category"] = df["category"].apply(process_category)
df["brand"] = df["brand"].apply(process_title_brand)

In [114]:
df.sample(5)

,category,description,title,brand,feature,rank,date,asin,imageURL,imageURLHighRes,also_view,price,fit,also_buy,main_cat,tech1,details,similar_item,tech2
59212,"[Clothing, Shoes & Jewelry, Women, Jewelry, Ne...",This dainty solitaire pendant showcases a sing...,14k White Gold Black Diamond Solitaire Pendant...,Amazon Collection,[Black diamonds may have been treated to impro...,"5,936,095inClothing,ShoesJewelry(",Amazon Collection,B000U8ID2O,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4231,"[Clothing, Shoes & Jewelry, Women, Jewelry, Br...",You'll love the way this exceptional bracelet ...,25 TCW Oval-Cut Genuine Garnet 14k Yellow Gold...,Palm Beach Jewelry,[FREE SHIPPING on orders of $30.00 or more - T...,"24,038,496inClothing,ShoesJewelry(",Palm Beach Jewelry,B00007FRKV,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49394,"[Clothing, Shoes & Jewelry, Girls, Jewelry, Ea...",None,Sterling Silver Claddagh Ring Shepherds Hook E...,SilverBin,"[Hand Crafted Sterling Silver, Celtic Design, ...","24,340,706inClothing,ShoesJewelry(",SilverBin,B000P9EZKM,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25218,"[Clothing, Shoes & Jewelry, Women, Shoes, Pump...","For the latest in high fashion footwear, Vanel...",VANELi Women's Caralyn Low-Heel Pump,VANELi,"[100% Suede, Synthetic sole, Heel measures app...","25,329,070inClothing,ShoesJewelry(",5 star,B000G0GGKW,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31340,"[Clothing, Shoes & Jewelry, Women, Handbags & ...",All canvas bags are not created equal! Certifi...,Organic Canvas Tote Bag,Ecobags,"[A certified organic 10 oz cotton tote, 19 (in...","2,538,770inClothing,ShoesJewelry(",5 star,B000I2X36S,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,"[B004V50P7S, B076XSGSKQ, B000I2PYYC, B07FPYVJW...",$12.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [120]:
randon_index = random.randint(0, chunk_size)
print(randon_index)
df["feature"].iloc[randon_index]

44059


['Measures 16" x 13" x 1.5"',
 'Two zippered compartments with flap closure',
 'Windowed handle pulls through flap',
 'Padded laptop sleeve',
 'Detachable shoulder strap with AirLift pad']

In [116]:
df.iloc[64528]["category"]

['Clothing, Shoes & Jewelry',
 'Women',
 'Shoes',
 'Flats',
 '100% Manmade',
 'Imported',
 'Synthetic sole',
 'Heel measures approximately 1"',
 'Platform measures approximately 0.25"']

In [105]:
df[df["category"].apply(lambda x: len(x) > 10)][["category", "feature"]].sample(3)

,category,feature
55388,"[Clothing, Shoes & Jewelry, Men, Shoes, Boots,...","[Leather, Shaft measures approximately Ankle"" ..."
46383,"[Clothing, Shoes & Jewelry, Men, Shoes, Athlet...","[100% Nylon, NoSeam in the upper provides a so..."
48290,"[Clothing, Shoes & Jewelry, Novelty & More, Cl...",[Screen printed graphic with a classic casual ...


In [ ]:
for i in range(0, len(dataset), chunk_size):
    df = dataset.load_jsonl_to_df(i, i + chunk_size)
    print(df)
    break